This notebook trains NeuralAI's Mamba family with supervised fine-tuning (SFT LoRA).
It intentionally skips `mamba-ssm` / `causal-conv1d` (which create Colab dependency conflicts) and relies on the Transformers sequential Mamba fallback.
Training is slower but fully functional and stable on stock Colab.


Make sure a GPU is attached. If `torch.cuda.is_available()` is False, training will be too slow.


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


Install torch (cu121), transformers, peft, accelerate, datasets, and llama-cpp-python for later quantization.


In [ ]:
!pip install -q --upgrade pip wheel
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.45 peft>=0.15 accelerate>=1.0 datasets sentencepiece tiktoken huggingface_hub
!pip install -q llama-cpp-python
import transformers, peft, torch
print('transformers', transformers.__version__, '| peft', peft.__version__, '| torch', torch.__version__)


Login reads/writes your HF namespace.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


Optional Google Drive mount keeps checkpoints between sessions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/Subject-Emu-5259/NeuralAI.git
%cd NeuralAI
!mkdir -p checkpoints models


We use the vocabulary-friendly 'intel' prompt format that only requires existing tokens (no `<|im_start|>` or `</s>` special tokens).


In [ ]:
!python3 scripts/prepare_sft_data.py --input data/train_sft_ultrachat_1k.jsonl --output data/train_intel_ultrachat_1k.jsonl --format intel
!python3 scripts/prepare_sft_data.py --input data/train_sft_ultrachat_10k.jsonl --output data/train_intel_ultrachat_10k.jsonl --format intel
!wc -l data/train_intel_ultrachat_*.jsonl
!head -1 data/train_intel_ultrachat_1k.jsonl


500 steps, batch 1, grad accum 4. ~15-30 min on T4 in sequential fallback.


In [ ]:
!python3 training/train_mamba_lora.py \
  --base models/mamba-k1 \
  --data data/train_intel_ultrachat_1k.jsonl \
  --output_dir checkpoints \
  --run_name k1-lora-sft-v2 \
  --max_steps 500 --warmup_steps 50 --batch_size 1 --grad_accum 4 --lr 5e-5 \
  --save_every 50 --log_every 10


Download base first if absent. 500 steps, grad accum 8.


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('state-spaces/mamba-790m-hf', local_dir='models/mamba-k2-base', local_dir_use_symlinks=False)
!python3 training/train_mamba_lora.py \
  --base models/mamba-k2-base \
  --data data/train_intel_ultrachat_10k.jsonl \
  --output_dir checkpoints \
  --run_name k2-lora-sft \
  --max_steps 500 --warmup_steps 50 --batch_size 1 --grad_accum 8 --lr 5e-5 \
  --save_every 100 --log_every 10


Download base first. 1000 steps, grad accum 16. Reduce steps if VRAM is tight.


In [ ]:
!python3 -c "from huggingface_hub import snapshot_download; snapshot_download('state-spaces/mamba-2.8b-slimpj', local_dir='models/mamba-k3-base', local_dir_use_symlinks=False)"
!python3 training/train_mamba_lora.py \
  --base models/mamba-k3-base \
  --data data/train_intel_ultrachat_10k.jsonl \
  --output_dir checkpoints \
  --run_name k3-lora-sft \
  --max_steps 1000 --warmup_steps 100 --batch_size 1 --grad_accum 16 --lr 2e-5 \
  --save_every 100 --log_every 10


Merge the trained LoRA adapter into the base model.


In [ ]:
!python3 scripts/merge_and_export.py \
  --base models/mamba-k1 \
  --adapter checkpoints/k1-lora-sft-v2/final \
  --output models/mamba-k1-merged-v2 \
  --hub_model_id Subject-Emu-5259/NeuralAI-Mamba-K1-v2


Quantize the merged model for local inference.


In [ ]:
!test -f llama.cpp/gguf-py/gguf_convert.py || git clone https://github.com/ggerganov/llama.cpp
!python3 llama.cpp/convert_hf_to_gguf.py models/mamba-k1-merged-v2 --outfile models/mamba-k1-merged-v2-f16.gguf

# Or use llama.cpp CLI quantize after building from source


- **OOM**: reduce `--max_steps`, increase `--grad_accum`, or enable gradient checkpointing in the training script.
- **Slow**: expected without mamba-ssm. Install only if your torch/CUDA are compatible: `pip install mamba-ssm causal-conv1d --no-build-isolation`.
- **HF auth**: make sure your token has write access to the target repo.
